# Developed Markets Validation

This notebook provides external validation of the already-selected US specification.

The fixed 50/50 model is **not reselected or reweighted** in foreign markets. Only its two component specifications are trained within each country:

- `LGBM_40`
- `DEEPSET_40_DYNAMIC`

Their OOS forecasts are then averaged mechanically with fixed 50/50 weights.

The four pre-specified developed markets are the United Kingdom, Australia, Germany, and France.


## 1. Runtime and project setup


In [ ]:
import os
import sys
import importlib
from pathlib import Path

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/FDS")
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "jkp_developed_153_parquet_2000_2024"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
PROJECT_DIR = DRIVE_PROJECT_DIR
DATA_DIR = DRIVE_DATA_DIR
RUNTIME = "Google Colab kernel"

for required in (PROJECT_DIR, PROJECT_DIR / "src", DATA_DIR):
    if not required.is_dir():
        raise FileNotFoundError(f"Required directory not found: {required}")

os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)

for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

importlib.invalidate_caches()

import src

src_file = Path(src.__file__).resolve()
if PROJECT_DIR.resolve() not in src_file.parents:
    raise RuntimeError(f"Imported src from the wrong location: {src_file}")

print("Runtime:", RUNTIME)
print("Project directory:", PROJECT_DIR)
print("Country data directory:", DATA_DIR)


## 2. Frozen country and model configuration

Each country keeps the same 15-year training, 4-year validation, 1-year test design. With data beginning in 2000, the OOS period is 2019–2024.

Only the two frozen component specifications are estimated. The fixed 50/50 is constructed after those OOS predictions exist.


In [ ]:
from src.config import ExperimentConfig, UniverseConfig

COUNTRIES = ("GBR", "AUS", "DEU", "FRA")
COUNTRY_NAMES = {
    "GBR": "United Kingdom",
    "AUS": "Australia",
    "DEU": "Germany",
    "FRA": "France",
}

COMPONENT_MODELS = (
    "LGBM_40",
    "DEEPSET_40_DYNAMIC",
)

OUTPUT_DIR = PROJECT_DIR / "model_runs" / "developed_markets"
SUMMARY_DIR = OUTPUT_DIR / "summary"

CONFIGS = {
    country: ExperimentConfig(
        experiment_id=f"external_validation_{country}_v1",
        project_dir=PROJECT_DIR,
        data_path=DATA_DIR / f"jkp_{country}_153_2000_2024.parquet",
        output_dir=OUTPUT_DIR,
        selected_models=COMPONENT_MODELS,
        seed=42,
        use_gpu=True,
        universe=UniverseConfig(
            country=country,
            start_year=2000,
            end_year=2024,
            security_id_col="id",
        ),
    )
    for country in COUNTRIES
}

for config in CONFIGS.values():
    config.validate()
    if not config.data_path.is_file():
        raise FileNotFoundError(config.data_path)

for country, config in CONFIGS.items():
    print(country, COUNTRY_NAMES[country], "->", config.run_dir)


## 3. Run or load the two component models in each country

The same component specifications used in the US chosen model are estimated independently within each foreign market. 


In [ ]:
import pandas as pd

from src.runner import ExperimentRunner

country_component_tables = {}

for country, config in CONFIGS.items():
    print(f"\n{'=' * 70}")
    print(country, "-", COUNTRY_NAMES[country])
    print("=" * 70)

    runner = ExperimentRunner(config)
    country_component_tables[country] = runner.run()

print("All component runs complete.")


## 4. Construct the fixed 50/50 model in each country




In [ ]:
from src.ensemble import ENSEMBLE_ID, build_fixed_fifty_fifty

country_fifty_fifty_metrics = {}

for country, config in CONFIGS.items():
    country_fifty_fifty_metrics[country] = build_fixed_fifty_fifty(config)
    print(country, "fixed 50/50 complete")

pd.DataFrame(
    [
        {"country": country, **metrics}
        for country, metrics in country_fifty_fifty_metrics.items()
    ]
)


## 5. Country-level comparison 




In [ ]:
from src.developed_markets import country_validation_table

country_tables = {}

for country, config in CONFIGS.items():
    table = country_validation_table(
        config,
        build_chosen_model=False,
    )
    country_tables[country] = table

    print(f"\n{country} — {COUNTRY_NAMES[country]}")
    display(table)


## 6. Summary developed-market validation




In [ ]:
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

developed_markets_validation = pd.concat(
    [country_tables[country] for country in COUNTRIES],
    ignore_index=True,
)

country_order = {country: i for i, country in enumerate(COUNTRIES)}
model_order = {
    ENSEMBLE_ID: 0,
    "LGBM_40": 1,
    "DEEPSET_40_DYNAMIC": 2,
}

developed_markets_validation = (
    developed_markets_validation
    .assign(
        _country_order=lambda frame: frame["country"].map(country_order),
        _model_order=lambda frame: frame["model_id"].map(model_order),
    )
    .sort_values(["_country_order", "_model_order"])
    .drop(columns=["_country_order", "_model_order"])
    .reset_index(drop=True)
)

developed_markets_validation.to_csv(
    SUMMARY_DIR / "developed_markets_validation.csv",
    index=False,
)

developed_markets_validation


## 7. Chosen-model external-validation summary

This final table isolates the fixed 50/50 result across the four countries.


In [ ]:
chosen_external = (
    developed_markets_validation.loc[
        developed_markets_validation["model_id"].eq(ENSEMBLE_ID)
    ]
    .reset_index(drop=True)
)

chosen_external


## 8. Final external-validation audit

In [ ]:
expected_models = {ENSEMBLE_ID, 'LGBM_40', 'DEEPSET_40_DYNAMIC'}
if len(developed_markets_validation) != 12:
    raise RuntimeError(f'Expected 12 country-model rows, found {len(developed_markets_validation)}')
if set(developed_markets_validation['country']) != set(COUNTRIES):
    raise RuntimeError('Developed-market country coverage is incomplete.')
if set(developed_markets_validation['model_id']) != expected_models:
    raise RuntimeError('Developed-market model coverage is incomplete.')
if len(chosen_external) != 4:
    raise RuntimeError('Expected one chosen-model result per country.')
print('EXTERNAL-VALIDATION AUDIT PASS')
